In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import _canoeValues as cv
import math
import time

#-------------------Inputs-------------------------------------------------------------
script = "Longitudinal Analysis_2 Paddler.ipynb"
input_file = "Station Information.csv"
outerHull_file = "Outer Hull.csv" # Will be used for hydrostatic force and waterline calculation
#--------------------------------------------------------------------------------------
script_dir = os.path.abspath('data') # absolute file path in Jupyter
file_path = os.path.join(script_dir, input_file) # file path for the station information

# Get the canoe's volume and mass from the station info
canoe_info = pd.read_csv(file_path, header=None)
statVol = canoe_info[1][1:]
statMass = canoe_info[2]
distNextStation = [float(x) for x in canoe_info[4][1:]]

# Get the outer hull coordinates from the input file
file_path = os.path.join(script_dir, outerHull_file)
out_coor = pd.read_csv(file_path, header=None)
x_out = out_coor[0]
y_out = out_coor[1]
z_out = out_coor[2]

station_no = x_out.nunique() # number of stations
station = x_out.unique() # location of each station

waterline_file = "Waterline_2 Paddlers.csv"

file_path = os.path.join(script_dir, waterline_file) # file path for the station information
waterline_info = pd.read_csv(file_path, header=None)
waterline_z = [float(x) for x in waterline_info[1][1:]]
print(waterline_z)
vol_displaced_per_station = [float(x) for x in waterline_info[2][1:]]

inner_file = "Inner Hull.csv"
outer_file = "Outer Hull.csv"

file_path = os.path.join(script_dir, inner_file) # file path for the text file input
in_coor = pd.read_csv(file_path, header=None)

file_path = os.path.join(script_dir, outer_file) # file path for the text file input
out_coor = pd.read_csv(file_path, header=None)

# Get inner and outer hull coordinates (station points)
x_in = in_coor[0]
y_in = in_coor[1]
z_in = in_coor[2]

x_out = out_coor[0]
y_out = out_coor[1]
z_out = out_coor[2]

station_no = x_out.nunique() # number of stations
station_out = x_out.unique() # location of each station
station_in = x_in.unique()


file_path = os.path.join(script_dir, "Length_vs_Moment_2 Paddler.csv") # file path for 2 pad moments
momentByLength = pd.read_csv(file_path, header=None)
momentArray = []
for i in range(len(station_in)):
    momentArray.append(momentByLength[2][i * round(len(momentByLength[0])/len(station_in)) + 1])


In [ ]:
# Figure out how to import waterline points, then interpolate exterior hull points to get x
canoe_depth = max(abs(z_out)) 
I_per_station = []
stat_waterline = [canoe_depth - abs(waterline) for waterline in waterline_z]
for k in range(0,station_no-1):
    # Station k coordinates (x,y)
    X1_value = station[k]
    Y1 = []
    Z1 = []
    
    # Station k+1 coordinates (x,y)
    X2_value = station[k+1]
    Y2 = []
    Z2 = []

    Y1_interp = []
    Y2_interp = []

    for i in range(0,len(x_out),1):
        # Get the points under the waterline
        if x_out[i] == X1_value:
            if canoe_depth - abs(z_out[i]) <= stat_waterline[k]:
                Y1.append(y_out[i])
                Z1.append(z_out[i])
                
                if canoe_depth - abs(z_out[i-1]) > stat_waterline[k]:

                    yInterpPoint = y_out[i-1] + ((y_out[i]-y_out[i-1]) / (z_out[i] - z_out[i-1])) * (stat_waterline[k] - canoe_depth - z_out[i-1])
                    
                    Y1_interp.append(yInterpPoint)
                            
                if canoe_depth - abs(z_out[i+1]) > stat_waterline[k]:
                    
                    yInterpPoint = y_out[i] + ((y_out[i+1]-y_out[i]) / (z_out[i+1] - z_out[i])) * (stat_waterline[k] - canoe_depth - z_out[i])
                    
                    Y1_interp.append(yInterpPoint)

        elif x_out[i] == X2_value:
            if canoe_depth - abs(z_out[i]) <= stat_waterline[k+1]:
                Y2.append(y_out[i])
                Z2.append(z_out[i])
                
                if canoe_depth - abs(z_out[i-1]) > stat_waterline[k+1]:

                    yInterpPoint = y_out[i-1] + ((y_out[i]-y_out[i-1]) / (z_out[i] - z_out[i-1])) * (stat_waterline[k+1] - canoe_depth - z_out[i-1])
                    
                    Y2_interp.append(yInterpPoint)
                            
                if canoe_depth - abs(z_out[i+1]) > stat_waterline[k+1]:
                    
                    yInterpPoint = y_out[i] + ((y_out[i+1]-y_out[i]) / (z_out[i+1] - z_out[i])) * (stat_waterline[k+1] - canoe_depth - z_out[i])
                    
                    Y2_interp.append(yInterpPoint)
                            
    if len(Y1) == 0 and len(Y2) == 0:
        # Station is out of the water
        continue
    elif len(Y1) == 0 or len(Y2) == 0:
        # Interpolate the waterline for the submerged station
        if len(Y1) == 0:
            Y2 = [Y2_interp[0]] + Y2 + [Y2_interp[-1]]
            Z2 = [stat_waterline[k+1] - canoe_depth] + Z2 + [stat_waterline[k+1] - canoe_depth]
            
            Y1 = [0]
            Z1 = [stat_waterline[k]]
            
            # Construct a rectangle using the average y values of the two stations
            b = distNextStation[k]
            h = abs(Y2_interp[0] * 2)
            I_per_station.append((b * h**3)/12)
                        
        else:
            Y1 = [Y1_interp[0]] + Y1 + [Y1_interp[-1]]
            Z1 = [stat_waterline[k] - canoe_depth] + Z1 + [stat_waterline[k] - canoe_depth]
            
            Y2 = [0]
            Z2 = [stat_waterline[k+1]]
            
            # Construct a rectangle using the average y values of the two stations
            b = distNextStation[k]
            h = abs(Y1_interp[0] * 2)
            I_per_station.append((b * h**3)/12)
    else:
        Y1 = [Y1_interp[0]] + Y1 + [Y1_interp[-1]]
        Y2 = [Y2_interp[0]] + Y2 + [Y2_interp[-1]]
        
        Z1 = [stat_waterline[k] - canoe_depth] + Z1 + [stat_waterline[k] - canoe_depth]
        Z2 = [stat_waterline[k+1] - canoe_depth] + Z2 + [stat_waterline[k+1] - canoe_depth]
    
        # Construct a rectangle using the average y values of the two stations
        b = distNextStation[k]
        h = abs((Y1_interp[0] * 2 + Y2_interp[0] * 2) / 2)
        I_per_station.append((b * h**3)/12)
        
    if len(Y1) != len(Y2):
        # Add points at the waterline to ensure both stations have THE SAME NUMBER OF POINTS below the waterline
        # This is necessary to ensure we are calculating the volume of the correct shape (points are not misaligned)
        while len(Y1) < len(Y2):
            Y1 = [Y1[0]] + Y1 + [Y1[-1]]
            Z1 = [Z1[0]] + Z1 + [Z1[-1]]
        while len(Y2) < len(Y1):
            Y2 = [Y2[0]] + Y2 + [Y2[-1]]
            Z2 = [Z2[0]] + Z2 + [Z2[-1]]
    
    # Depth and Moment (Value) for each station
    depth = (abs(min(Z1))+abs(min(Z2)))/2
    value = float(momentArray[k])
    
    # Calculate the canoe's volume (intermediate step for y-bar)
    volume = 0
    product = 0 # y_bar*Volume
    for i in range(0,len(Y1)-1,1):
        z_i = ((Z1[i]+Z1[i+1])/2 + (Z2[i]+Z2[i+1])/2) /2
        x = (X1_value + X2_value)/2
        y = (abs(Y1[i]) + abs(Y1[i+1]) + abs(Y2[i]) + abs(Y2[i+1]))/4
        z = (abs(Z1[i+1]-Z1[i]) + abs(Z2[i+1]-Z2[i]))/2

        volume_i = x * y * z
        volume += volume_i
        product += volume_i * z_i
    
    # Calculate y-bar from the top and bottom
    if volume > 0:
        y_bar_top = abs(product/volume)
        y_bar_bottom = y_bar_top - depth
        
        total_volume += volume
        acc += y_bar_bottom * volume
    else:
        pass
    
center_of_buoyancy = abs(acc / total_volume)

# Moment of inertia
I = sum(I_per_station)
V = sum(vol_displaced_per_station)

buoyancy_to_metacenter = I/V

print("Center of Buoyancy:", center_of_buoyancy, "mm")
print("Moment of Inertia:", I, "mm^4")
print("Volume of Water Displaced:", V, "mm^3")
print("Buoyancy to Metacenter:", buoyancy_to_metacenter, "mm")


In [ ]:
acc = 0
total_volume = 0
# Loop through each station
for k in range(1,station_no-2,1):
    # print(k)

    # X-coordinates for the sides of this station
    X1_in = station_in[k]
    X2_in = station_in[k+1]
    X1_out = station_out[k]
    X2_out = station_out[k+1]

    # Arrays to store the YZ-coordinates of each point in the station (inner and outer hull)
    Y1_in = []; Z1_in = []
    Y2_in = []; Z2_in = []
    Y1_out = []; Z1_out = []
    Y2_out = []; Z2_out = []

    # Loop through each x coordinate and get every point in the cross-section
    for i in range(0,len(x_in),1):

        if x_out[i] == X1_out:
            Y1_out.append(y_out[i])
            Z1_out.append(z_out[i])

        elif x_out[i] == X2_out:
            Y2_out.append(y_out[i])
            Z2_out.append(z_out[i])
        
        if x_in[i] == X1_in:
            Y1_in.append(y_in[i])
            Z1_in.append(z_in[i])

        elif x_in[i] == X2_in:
            Y2_in.append(y_in[i])
            Z2_in.append(z_in[i])

    # Depth and Moment (Value) for each station
    depth = (abs(min(Z1_out))+abs(min(Z2_out)))/2
    value = float(momentArray[k])


    # Calculate the canoe's volume (intermediate step for y-bar)
    volume = 0
    product = 0 # y_bar*Volume
    for i in range(0,len(Y1_out)-1,1):
        z_i = ((Z1_out[i]+Z1_out[i+1]+Z1_in[i]+Z1_in[i+1])/4 + (Z2_out[i]+Z2_out[i+1]+Z2_in[i]+Z2_in[i+1])/4) /2
        x = (abs(X1_in-X2_in) + abs(X1_out-X2_out))/2
        y = (abs(Y1_in[i] - Y1_out[i]) + abs(Y1_in[i+1]-Y1_out[i+1]) + abs(Y2_in[i] - Y2_out[i]) + abs(Y2_in[i+1]-Y2_out[i+1]))/4
        z = (abs(Z1_out[i+1]-Z1_out[i]) + abs(Z1_in[i+1]-Z1_in[i]) + abs(Z2_out[i+1]-Z2_out[i]) + abs(Z2_in[i+1]-Z2_in[i]))/4

        volume_i = x * y * z
        volume += volume_i
        product += volume_i * z_i
    
    # Calculate y-bar from the top and bottom
    y_bar_top = abs(product/volume)
    y_bar_bottom = y_bar_top - depth
    
    total_volume += volume
    acc += y_bar_bottom * volume
    
center_of_gravity = abs(acc / total_volume)
print("Center of Gravity:", center_of_gravity, "mm")


In [ ]:
metacentric_height = (buoyancy_to_metacenter + center_of_buoyancy) - center_of_gravity
print("Metacentric Height:", metacentric_height, "mm")